# NexPlay · modelos de texto (02)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fernandoaxelramirezgomez-coder/nexplay/blob/master/notebooks/02_modelos_texto.ipynb)
[![Ver en nbviewer](https://img.shields.io/badge/ver%20en-nbviewer-orange)](https://nbviewer.org/github/fernandoaxelramirezgomez-coder/nexplay/blob/master/notebooks/02_modelos_texto.ipynb)

**Objetivo.** Responder, con una regla fijada antes de correr, si el texto de las reseñas negativas distingue
las tempranas (escritas antes de 120 minutos de juego, `Y = 1`) de las tardías. Si las distingue, las quejas
del arrepentimiento temprano tienen lenguaje propio; si no, la ficha presenta sus motivos como quejas generales.

**Entradas.** El código se clona del tag `codigo-v6` de este repositorio (importa `backend/analisis/texto.py`,
`backend/analisis/limpieza.py` y la partición congelada de `backend/referencias/`; la sección 6 importa además
`backend/analisis/lectura_texto.py`). Los datos se bajan de un
release con tag fijo y el asset se verifica con su sha256 antes de abrirlo:

| Release | Asset | sha256 | Para qué |
|---|---|---|---|
| `data-v1` | `nexplay_reproducible.db.xz` | `2ef8ef40330385af4c03cd072dccb20fc9a4b635e3929e513235c191d14e9ee7` | 83 juegos: las negativas en inglés del conjunto limpio |

Baja además el modelo de embeddings `sentence-transformers/all-MiniLM-L6-v2`, revisión
`1110a243fdf4706b3f48f1d95db1a4f5529b4d41` (unos 90 MB). La regla está en
`docs/evidencia/modelos-texto-prerregistro.md` (commit `6bbcea6`) y el resultado registrado, en
`docs/evidencia/modelos-texto.json`.

**Salidas.** Tablas y gráficas dentro del notebook. En la carpeta de trabajo quedan el clon (`repo_nexplay/`), la base
bajada (`datos_texto/`) y los vectores de MiniLM (`extracto_texto/`); el modelo de embeddings queda en la caché
de Hugging Face. No escribe nada en el repositorio.

**Cómo correrlo.** En Colab, con el botón de arriba y *Entorno de ejecución → Ejecutar todas*. En local, desde
la raíz del repositorio, `make notebooks` ejecuta este, el 00 y el 01 con el código de tu checkout en lugar del
tag.

**Tiempo estimado.** Unos 8 minutos en local (491 s medidos en un i7 de 8 núcleos, con el modelo de
embeddings ya en caché; la primera vez baja además unos 90 MB). Con las versiones de Colab, 400 s (medido con las
mismas versiones de Colab, no en Colab). Lo que más tarda es codificar las 16,836 reseñas con MiniLM y el bootstrap.

Corrió en Colab el 2026-10-01 con `codigo-v4`, sin errores ni avisos (Colab con T4 disponible; los embeddings
corren en CPU); evidencia en `docs/evidencia/colab/`. Con `codigo-v6`, que agrega la lectura para negocio de la
sección 6, falta correrlo en Colab. Las salidas guardadas aquí son de una corrida local, para que las gráficas de
la sección 6 se vean también como PNG en GitHub.

Cómo leerlo:

- **La pregunta.** Entre las reseñas negativas, ¿el texto distingue las tempranas de las tardías (negativas
  escritas con 120 minutos de juego o más)? Si sí, las quejas del arrepentimiento temprano tienen lenguaje propio
  y los motivos de la ficha son específicos de ese momento. Si no, la ficha los presenta como quejas generales.
- **Prerregistro.** Los datos, la máscara de duraciones, los cinco modelos, la métrica y la regla de decisión se
  fijaron y se commitearon antes de escribir el código. Aquí solo se calcula y se aplica la regla. La última
  celda de código comprueba que las cifras coinciden con las registradas.
- **Una señal, no una confirmación.** Una reseña temprana es una **señal de arrepentimiento temprano**: no
  confirma que alguien se arrepintiera.
- **El texto no entra al score de riesgo**, porque `Y` sale de esas mismas reseñas. Ningún modelo de este
  notebook llega a la API.
- Las cuentas viven en `backend/analisis/texto.py`; este notebook las importa. Las cifras las calcula cada
  celda; el texto describe lo que se ve, y un `assert` en la celda de arriba falla si los datos dejan de
  sostenerlo.

## 0. Preparación

In [1]:
# Colab ya trae pandas, numpy, scikit-learn, plotly y torch: no se reinstalan, porque cambiar sus versiones rompe
# sus binarios. Solo se instala lo que falte. lingua va con versión fija: de ella depende la firma del conjunto
# limpio.
import importlib.util
import subprocess
import sys

requeridos = {"plotly": "plotly", "lingua": "lingua-language-detector==2.2.0", "sentence_transformers": "sentence-transformers",
              "wordcloud": "wordcloud==1.9.6"}
faltantes = [paquete for modulo, paquete in requeridos.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)

In [2]:
# Código y datos con referencias fijas: un tag de código y el release de datos con su sha256.
GITHUB_REPO = "fernandoaxelramirezgomez-coder/nexplay"
CODIGO_REF = "codigo-v6"
DATA_V1_SHA256 = "2ef8ef40330385af4c03cd072dccb20fc9a4b635e3929e513235c191d14e9ee7"  # 83 juegos

In [3]:
import os
import shutil

# Callado si sale bien: git avisa que el tag anotado «no es un commit» y que queda en «detached HEAD», y ninguna de
# las dos cosas es un error. Si el clon falla, se ve el mensaje de git.
# Un repo_nexplay que ya estaba se usa solo si es CODIGO_REF: un entorno de Colab que corrió otra versión conserva
# su clon, y con él faltarían los módulos nuevos. Sin .git es la copia que deja make notebooks y se usa tal cual.
if os.path.isdir("repo_nexplay/.git"):
    actual = subprocess.run(["git", "-C", "repo_nexplay", "describe", "--tags", "--exact-match"],
                            capture_output=True, text=True).stdout.strip()
    if actual != CODIGO_REF:
        print(f"repo_nexplay era {actual or 'otro código'}: se borra y se clona {CODIGO_REF}.")
        shutil.rmtree("repo_nexplay")
if os.path.isdir("repo_nexplay"):
    print("repo_nexplay ya existe, no se vuelve a clonar.")
else:
    clon = subprocess.run(
        ["git", "-c", "advice.detachedHead=false", "clone", "--quiet", "--branch", CODIGO_REF, "--depth", "1",
         f"https://github.com/{GITHUB_REPO}.git", "repo_nexplay"],
        capture_output=True, text=True,
    )
    if clon.returncode != 0:
        raise RuntimeError(f"no se pudo clonar {CODIGO_REF}: {clon.stderr.strip()}")

repo_nexplay era codigo-v5: se borra y se clona codigo-v6.


In [4]:
sys.path[:0] = ["repo_nexplay/backend", "repo_nexplay/backend/analisis", "repo_nexplay/backend/modelado"]

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import OptimizeWarning

import limpieza as li  # noqa: E402 (tras sys.path, a propósito)
import texto as tx  # noqa: E402
from despliegue.utilidades import descargar_verificado  # noqa: E402
from exploracion import cargar_release  # noqa: E402

assert tx.DATA_V1_SHA256 == DATA_V1_SHA256, "el código clonado espera otro data-v1"
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

# scikit-learn anterior a 1.7.1 (Colab trae la 1.6.1) le pasa `iprint` al L-BFGS-B de SciPy, que en las
# versiones nuevas ya no lo acepta y avisa en cada ajuste. El ajuste no cambia: solo se calla ese aviso.
warnings.filterwarnings("ignore", message="Unknown solver options: iprint", category=OptimizeWarning)


def con_intervalos(tabla: pd.DataFrame) -> pd.DataFrame:
    """Solo para mostrar: cifras con 4 decimales y cada intervalo como [bajo, alto]."""
    return tabla.map(lambda v: f"[{v[0]:.4f}, {v[1]:.4f}]" if isinstance(v, tuple) else ("" if pd.isna(v) else f"{v:.4f}"))

In [5]:
import sentence_transformers
import sklearn
import torch

print(f"python                {sys.version.split()[0]}")
print(f"numpy                 {np.__version__}")
print(f"pandas                {pd.__version__}")
print(f"scikit-learn          {sklearn.__version__}")
print(f"torch                 {torch.__version__}")
print(f"sentence-transformers {sentence_transformers.__version__}")

python                3.13.15
numpy                 2.1.3
pandas                2.2.3
scikit-learn          1.6.1
torch                 2.11.0+cu130
sentence-transformers 5.7.0


In [6]:
ruta_v1 = descargar_verificado("data-v1", DATA_V1_SHA256, Path("datos_texto") / "nexplay_data-v1.db")

descargando https://github.com/fernandoaxelramirezgomez-coder/nexplay/releases/download/data-v1/nexplay_reproducible.db.xz ...
descarga verificada (data-v1): 10.9 MB, sha256 OK
datos_texto/nexplay_data-v1.db reconstruida (41.3 MB)


## 1. Las reseñas

**Pregunta:** ¿qué reseñas entran y cuántas hay de cada clase?

Se parte del conjunto limpio del 00 (sin duplicados de 8 palabras o más ni reseñas vacías), con su misma firma.
De él quedan solo las negativas sin la marca de otro idioma, que es la definición del 00 de «en inglés». Las
cortas y las plantillas entran: se marcan, no se quitan.

In [7]:
_, resenas_v1 = cargar_release(ruta_v1)
limpio, _ = li.limpiar(resenas_v1)
firma = li.firma_del_conjunto(limpio)
assert firma == li.FIRMA_LIMPIO_DATA_V1, "el conjunto limpio no es el del 00: el prerregistro no aplica"
negativas = tx.negativas_en_ingles(limpio)
y, appid = negativas["temprana"].to_numpy(), negativas["appid"]
print(f"conjunto limpio: {len(limpio):,} reseñas, firma {firma[:12]}…")
print(f"negativas en inglés: {len(negativas):,} · tempranas {y.sum():,} ({y.mean():.2%}) · "
      f"tardías {(y == 0).sum():,} · {appid.nunique()} juegos")
# Los conteos del prerregistro (§2), medidos antes de cualquier modelo.
assert (len(negativas), y.sum(), appid.nunique()) == (16_836, 2_670, 83), "no son las reseñas del prerregistro"

conjunto limpio: 122,944 reseñas, firma cb1e4b06fdf5…
negativas en inglés: 16,836 · tempranas 2,670 (15.86%) · tardías 14,166 · 83 juegos


In [8]:
fold = tx.folds_de(negativas)
conteos = tx.conteos_por_fold(negativas, fold)
display(conteos.round(4))
assert conteos["tempranas"].tolist() == [606, 447, 624, 590, 403], "los folds no son los del prerregistro"

,juegos,negativas,tempranas,prevalencia
fold,,,,
0,16,3186,606,0.1902
1,16,2690,447,0.1662
2,17,3374,624,0.1849
3,17,4505,590,0.1310
4,17,3081,403,0.1308


### 1.1 La máscara de duraciones

La clase se define por los minutos jugados. Una reseña que dice cuánto jugó («refunded after 20 minutes»,
«300 hours in») repite la etiqueta: un modelo que la lee no encuentra lenguaje de la queja, lee la definición
de la clase. Por eso los cinco modelos leen el texto con cada duración cambiada por la palabra `duracion`, que
dice que hubo una, no cuál. El patrón está en `tx.PATRON_DURACION` y en el §3 del prerregistro: no se enmascaran
días ni semanas, ni una `m` pegada a una cifra («162m budget»).

In [9]:
con_duracion = negativas["texto_norm"].map(lambda t: tx.PATRON_DURACION.search(t) is not None)
print(f"negativas con al menos una duración enmascarada: {con_duracion.sum():,} ({con_duracion.mean():.1%})")
fragmentos = negativas.loc[con_duracion, "texto_norm"].sample(5, random_state=tx.SEMILLA).map(
    lambda t: t[max(0, tx.PATRON_DURACION.search(t).start() - 60): tx.PATRON_DURACION.search(t).end() + 60])
display(pd.DataFrame({"texto": fragmentos, "enmascarado": tx.enmascarar_duraciones(fragmentos)}).reset_index(drop=True))
assert not negativas["texto_modelo"].map(lambda t: tx.UNIDAD_SUELTA.search(t) is not None).any(), "quedó una unidad sin enmascarar"

# Lo que dice el texto de abajo: cerca de una de cada siete negativas dice cuánto jugó.
assert 0.12 < con_duracion.mean() < 0.17, "cambió la parte de las negativas con una duración"

negativas con al menos una duración enmascarada: 2,442 (14.5%)


,texto,enmascarado
0,ur head do not worry though you'll meet someone maybe every twenty hours of playtime nobody else is playing this burning pile a lot,ur head do not worry though you'll meet someone maybe every duracion of playtime nobody else is playing this burning pile a lot
1,lly needs to implement hardness levels all the players with 5000 hrs in the game complain is too easy while casual players that,lly needs to implement hardness levels all the players with duracion in the game complain is too easy while casual players that
2,got banned for god knows nothing got on played one lil minute of a dumb server took a 3 month break or more and come back,got banned for god knows nothing got on played one lil duracion of a dumb server took a 3 month break or more and come back
3,their family lives right down the hall i have put countless hours into the pre steam version and will continue to play it on,their family lives right down the hall i have put countless duracion into the pre steam version and will continue to play it on
4,game randomly freezes for 1 15 seconds randomly only happens on my 50 series rtx pc please fix,game randomly freezes for duracion randomly only happens on my 50 series rtx pc please fix


**Qué vemos.** Las tempranas son cerca de una de cada seis negativas en inglés, y su proporción cambia de un
fold a otro: por eso el trivial se mide en cada fold. Cerca de una de cada siete negativas dice cuánto jugó;
la máscara cambia esas menciones por `duracion` y no deja unidades sueltas.

## 2. Los modelos

**Pregunta:** ¿qué tan bien separa cada modelo a las tempranas de las tardías, en juegos que no vio?

Los cinco leen el mismo texto enmascarado y se validan con la partición congelada por `appid`: cada fold
predice reseñas de juegos que no estuvieron en su entrenamiento. Todo ajuste (vocabulario, idf, escalado y
coeficientes) ocurre dentro de un `Pipeline`, solo con el fold de entrenamiento. Los parámetros son los del
prerregistro, sin búsqueda:

1. **Trivial:** un puntaje constante. Su PR-AUC en cada fold es la prevalencia de ese fold.
2. **Regla «refund»:** 1 si la reseña contiene «refund» (refunded, refunding…), 0 si no. La palabra salió de la
   sección 3.7 del 00, sobre estos mismos datos: es una referencia optimista.
3. **TF-IDF + Naive Bayes:** palabras y pares de palabras que aparecen en 5 reseñas o más, sin stopwords.
4. **TF-IDF + regresión logística:** el mismo vocabulario, con clases balanceadas.
5. **all-MiniLM-L6-v2 + regresión logística:** vectores de un modelo de lenguaje preentrenado y congelado (no se
   ajusta con estos datos, así que se calculan una vez), escalados y con la misma regresión.

In [10]:
scores = {"trivial": np.zeros(len(y)), "refund": tx.regla_refund(negativas["texto_modelo"])}
menciona = pd.Series(scores["refund"]).groupby(np.where(y == 1, "temprana", "tardía")).mean()
display((100 * menciona).round(1).rename("% que menciona «refund»").to_frame())

# Lo que dice el texto de la sección: las tempranas lo mencionan más que las tardías, pero la mayoría no lo menciona.
assert menciona["tardía"] < menciona["temprana"] < 0.5, "cambió cómo mencionan «refund» las dos clases"

,% que menciona «refund»
tardía,2.9
temprana,9.6


In [11]:
# Con máscara, los que deciden; sin máscara, solo la sensibilidad de la sección 4.
for nombre, modelo in tx.modelos_tfidf().items():
    scores[nombre] = tx.scores_fuera_de_fold(modelo, negativas["texto_modelo"], y, appid)
    scores[f"{nombre}_sin"] = tx.scores_fuera_de_fold(modelo, negativas["texto_norm"], y, appid)

In [12]:
# La parte lenta: unos 9 minutos en CPU la primera vez. Los vectores quedan en extracto_texto/ y una segunda
# corrida los lee de ahí.
vectores = tx.embeddings(negativas["texto_modelo"], Path("extracto_texto"))
print(f"vectores: {vectores.shape[0]:,} reseñas × {vectores.shape[1]} dimensiones")
scores["minilm_lr"] = tx.scores_fuera_de_fold(tx.modelo_embeddings(), vectores, y, appid)

vectores: 16,836 reseñas × 384 dimensiones


In [13]:
por_fold = {nombre: tx.pr_auc_por_fold(y, score, fold) for nombre, score in scores.items()}
modelos = ["trivial", "refund", *tx.CANDIDATOS]
display(pd.DataFrame({tx.NOMBRES[n]: por_fold[n] for n in modelos}).rename_axis("fold").round(4))
assert np.allclose(por_fold["trivial"], conteos["prevalencia"]), "el trivial no es la prevalencia del fold"

# Lo que dice el texto de abajo, fold por fold.
assert (por_fold["refund"] > por_fold["trivial"]).all(), "la regla ya no supera al trivial en todos los folds"
assert all((por_fold[n] > por_fold["refund"]).all() for n in tx.CANDIDATOS), "un modelo ya no supera a la regla en todos los folds"
assert (por_fold["tfidf_lr"] > np.maximum(por_fold["tfidf_nb"], por_fold["minilm_lr"])).all(), "TF-IDF + LR ya no es el mejor en todos los folds"
assert (por_fold["minilm_lr"] > por_fold["tfidf_nb"]).all(), "MiniLM ya no queda segundo en todos los folds"
medias = {n: por_fold[n].mean() for n in modelos}
assert medias["tfidf_nb"] - medias["refund"] < medias["minilm_lr"] - medias["tfidf_nb"], "NB ya no queda más cerca de la regla"

,Trivial,Regla «refund»,TF-IDF + NB,TF-IDF + LR,MiniLM + LR
fold,,,,,
0,0.1902,0.2310,0.2584,0.4649,0.4056
1,0.1662,0.1879,0.2533,0.3780,0.3193
2,0.1849,0.2063,0.2549,0.3573,0.3507
3,0.1310,0.1400,0.1739,0.3097,0.2640
4,0.1308,0.1614,0.1917,0.3358,0.2753


**Qué vemos.** El orden se repite en los cinco folds. La regla «refund» queda por encima del trivial, y los tres
modelos quedan por encima de la regla. TF-IDF + LR está arriba de todos y MiniLM + LR, segundo. Naive Bayes
queda más cerca de la regla que de los otros dos modelos. Las tempranas mencionan el reembolso más que las
tardías, pero la mayoría no lo menciona: por eso una palabra sola separa poco.

## 3. Intervalos

**Pregunta:** ¿las diferencias se sostienen si se evalúan otros juegos?

Bootstrap sobre juegos, 2,000 réplicas con semilla 42. En cada réplica y en cada fold se sacan, con reemplazo,
tantos juegos como tiene el fold, y cada reseña pesa las veces que salió su juego. Las predicciones son las de
arriba: no se reentrena, así que el intervalo mide cuánto cambia la cifra según qué juegos se evalúan. Todas las
cifras de una réplica salen de los mismos juegos, de modo que las diferencias entre modelos son pareadas. IC del
95 % por percentiles.

- **cociente:** PR-AUC medio del modelo entre el del trivial (1 = no mejor que el azar).
- **− refund:** PR-AUC medio del modelo menos el de la regla «refund».

In [14]:
replicas = tx.bootstrap_pareado(y, scores, fold, appid.to_numpy())
tabla = tx.resumen(por_fold, replicas)
display(con_intervalos(tabla.rename(index=tx.NOMBRES)))

# Lo que dice el texto de abajo.
for nombre in tx.CANDIDATOS:
    assert tabla.loc[nombre, "IC cociente"][0] > 1, f"{nombre} ya no supera al trivial"
    assert tabla.loc[nombre, "IC − refund"][0] > 0, f"{nombre} ya no supera a «refund»"
assert tabla.loc["refund", "IC cociente"][0] > 1, "la regla ya no supera al trivial"
assert tabla.loc["refund", "IC cociente"][1] < tabla.loc["tfidf_nb", "IC cociente"][0], "la regla ya no queda por debajo de los modelos"
assert tabla.loc["tfidf_lr", "IC media"][1] - tabla.loc["tfidf_lr", "IC media"][0] > 0.1, "el intervalo ya no es ancho"

,PR-AUC media,std entre folds,IC media,cociente,IC cociente,− refund,IC − refund
Trivial,0.1606,0.0256,"[0.1229, 0.1952]",1.0000,"[1.0000, 1.0000]",,
Regla «refund»,0.1853,0.0321,"[0.1426, 0.2237]",1.1537,"[1.1057, 1.2327]",,
TF-IDF + NB,0.2264,0.0361,"[0.1770, 0.2749]",1.4098,"[1.3341, 1.5039]",0.0411,"[0.0234, 0.0564]"
TF-IDF + LR,0.3691,0.0530,"[0.2877, 0.4214]",2.2982,"[2.0453, 2.5311]",0.1838,"[0.1374, 0.2055]"
MiniLM + LR,0.3230,0.0517,"[0.2537, 0.3713]",2.0108,"[1.8294, 2.2014]",0.1377,"[0.1043, 0.1561]"


**Qué vemos.** Los intervalos son anchos: la cifra cambia bastante según qué juegos se evalúen. Aun así, ninguno
de los tres modelos cruza las líneas que importan para la regla. Su cociente contra el trivial queda por encima
de 1 y su diferencia con la regla «refund», por encima de 0. La regla sola también supera al trivial, pero queda
por debajo de los tres modelos, y la palabra se eligió mirando estos datos.

## 4. Sensibilidad: ¿cuánto ganaban por leer la duración?

NB y TF-IDF + LR, otra vez con el texto sin enmascarar, sobre las mismas reseñas y los mismos folds. La
ganancia es «sin máscara − con máscara», pareada. **No entra a la regla.**

In [15]:
sensibilidad = tx.sensibilidad(por_fold, replicas)
display(con_intervalos(sensibilidad.rename(index=tx.NOMBRES)))

# Lo que dice el texto de abajo: la duración ayudaba, pero poco frente a la ventaja sobre «refund».
for nombre in tx.SIN_MASCARA:
    assert sensibilidad.loc[nombre, "IC ganancia"][0] > 0, f"{nombre} ya no gana al leer la duración"
assert sensibilidad.loc["tfidf_lr", "ganancia"] < tabla.loc["tfidf_lr", "− refund"] / 5, "la ganancia por la duración ya no es chica"

,con máscara,sin máscara,ganancia,IC ganancia
TF-IDF + NB,0.2264,0.2301,0.0036,"[0.0019, 0.0050]"
TF-IDF + LR,0.3691,0.3818,0.0127,"[0.0070, 0.0173]"


**Qué vemos.** Leer la duración sí ayudaba: sin máscara, los dos modelos ganan en todo el intervalo. Pero la
ganancia es chica frente a lo que separa a TF-IDF + LR de la regla «refund». La ventaja del modelo no viene de
que la reseña diga cuánto se jugó. Lo que quede de esa señal en la marca `duracion` es un límite (ver la
conclusión).

## 5. Decisión

La regla del prerregistro (§7), aplicada por `tx.decidir` sobre el texto enmascarado:

1. Si el mejor modelo (M) no supera al trivial (el intervalo del cociente incluye 1), es una moneda: las quejas
   tempranas no tienen lenguaje propio.
2. Si M supera al trivial pero no a «refund» (la diferencia pareada incluye 0 o queda por debajo), un modelo no
   aporta más que una palabra: se descarta el modelo y se documenta la regla.
3. Si M supera a los dos, se guarda el modelo más simple cuya diferencia contra M incluye 0 y que también supera
   al trivial y a «refund».

En ningún caso entra al score de riesgo.

In [16]:
decision = tx.decidir(por_fold, replicas)
print(f"Mejor modelo (M): {tx.NOMBRES[decision['mejor']]}")
print(f"  supera al trivial: {decision['supera_trivial']} · supera a «refund»: {decision['supera_refund']}")
print(f"Rama {decision['rama']}" + (f": se guarda {tx.NOMBRES[decision['elegido']]}" if decision["elegido"] else ""))
print(decision["conclusion"])

registrado = json.loads(Path("repo_nexplay/docs/evidencia/modelos-texto.json").read_text(encoding="utf-8"))
diferencias = tx.diferencias_con_registro(registrado, por_fold, decision)
assert not diferencias, f"no coincide con docs/evidencia/modelos-texto.json: {diferencias}"
print(f"Coincide con docs/evidencia/modelos-texto.json (código {registrado['codigo']['commit']}).")

# El texto de la conclusión es el de esta rama.
assert (decision["rama"], decision["mejor"], decision["elegido"]) == (3, "tfidf_lr", "tfidf_lr"), "cambió la decisión: la conclusión de abajo ya no aplica"

Mejor modelo (M): TF-IDF + LR
  supera al trivial: True · supera a «refund»: True
Rama 3: se guarda TF-IDF + LR
M supera a los dos. Se guarda el modelo más simple cuya diferencia pareada contra M incluye 0 y que supera al trivial y a «refund». Conclusión: las quejas tempranas tienen lenguaje propio, más allá de «refund». Los motivos de la ficha se pueden presentar como propios de ese momento.
Coincide con docs/evidencia/modelos-texto.json (código 59694a6).


## Conclusión

**Rama 3 de la regla.** El mejor modelo es TF-IDF + regresión logística, y supera al trivial y a la regla
«refund». Los otros dos quedan por debajo de él con intervalos que no tocan el cero. Por eso el más simple que lo
iguala es él mismo, y se guarda TF-IDF + LR, como análisis.

Conclusión, copiada del prerregistro: **las quejas tempranas tienen lenguaje propio, más allá de «refund», y los
motivos de la ficha se pueden presentar como propios de ese momento.**

Lo que esto no dice:

- Es una **señal de arrepentimiento temprano**, no un arrepentimiento confirmado.
- Que el modelo distinga a las tempranas no dice qué motivos lo hacen. Las categorías de palabras clave de la
  ficha no se prueban aquí: eso es la Parte B. Las palabras que separan a los dos grupos están en la sección 3.7
  del 00.
- La máscara es una lista fija. Lo que se le escapa («a while», «days», «playtime») queda en el texto, y la sola
  presencia de `duracion` puede ser distinta entre las clases.
- Solo inglés y solo data-v1: los 40 juegos externos no se usan. El intervalo no incluye la variación que daría
  reentrenar.
- **No entra al score de riesgo**, porque `Y` sale de esas mismas reseñas. Ningún modelo de aquí se publica ni se
  sirve.

## 6. Lectura para negocio (posterior a la decisión)

**Descriptiva.** Esta sección se agregó después de la decisión de la sección 5 y no la cambia. Usa los mismos
puntajes fuera de fold, no entra a la regla ni al score de riesgo, y nada de aquí se fijó antes de correr. Sirve
para leer el resultado en términos de negocio: cuánto separa cada modelo, qué tan arriba quedan las tempranas si
se revisa solo una parte, qué palabras pesan y cómo se ven los aciertos y los errores.

In [17]:
# True en la versión que se sube al repo: las gráficas de esta sección se guardan también como PNG para que se vean
# en GitHub, con el mismo mecanismo que el 00. En Colab, o sin Chrome, se apaga sola con un aviso.
EXPORTAR_ESTATICO = True

import graficas  # noqa: E402
import lectura_texto as lt  # noqa: E402

EXPORTAR_ESTATICO = graficas.configurar_graficas(EXPORTAR_ESTATICO)

Exportación estática apagada: en Colab no hace falta. Las gráficas se ven interactivas, sin PNG.


### 6.1 Cuánto separa cada modelo

**Pregunta:** ¿qué tan lejos del azar queda cada modelo? Las barras son la media de los 5 folds con su IC del
95 % (la tabla de la sección 3); la línea punteada es el trivial.

In [18]:
lt.figura_pr_auc_con_ic(tabla).show()
# Lo que dice el texto de abajo.
assert tabla.loc["tfidf_lr", "PR-AUC media"] == tabla.loc[list(tx.CANDIDATOS), "PR-AUC media"].max()
assert tabla.loc["tfidf_lr", "IC media"][0] > tabla.loc["refund", "IC media"][1], "el intervalo de TF-IDF + LR ya toca al de la regla"
assert tabla.loc["minilm_lr", "IC media"][1] > tabla.loc["tfidf_lr", "IC media"][0], "las barras de MiniLM y TF-IDF + LR ya no se traslapan"
assert decision["mejor_menos_cada_uno"]["minilm_lr"][0] > 0, "en la comparación pareada MiniLM ya no queda por debajo"
assert (tabla.loc[["refund", *tx.CANDIDATOS], "PR-AUC media"] > tabla.loc["trivial", "PR-AUC media"]).all()

**Qué vemos.** TF-IDF + LR es la barra más alta, y su intervalo queda entero por encima del de la regla «refund».
Las barras de MiniLM y de TF-IDF + LR se traslapan, pero eso no las empata: la comparación que decide es pareada,
réplica por réplica, y ahí MiniLM queda por debajo (sección 5). Todas superan la línea del trivial.

### 6.2 Precisión contra recall

**Pregunta:** si se señalaran más o menos reseñas, ¿qué parte de lo señalado sería temprana? Cada curva es el
promedio de los 5 folds: para cada recall (qué parte de las tempranas se encuentra), la precisión (qué parte de lo
señalado es temprana). Su área se parece a la media de los PR-AUC de la tabla.

In [19]:
curvas = {nombre: lt.curva_pr_media(y, scores[nombre], fold) for nombre in ["trivial", "refund", *tx.CANDIDATOS]}
lt.figura_curvas_pr(curvas).show()
# Lo que dice el texto de abajo.
for nombre in ("refund", "tfidf_nb", "minilm_lr"):
    assert (curvas["tfidf_lr"] >= curvas[nombre]).mean() > 0.95, f"TF-IDF + LR ya no queda arriba de {nombre} en casi todo el recall"
assert abs(curvas["tfidf_lr"].mean() - por_fold["tfidf_lr"].mean()) < 0.01, "la curva media ya no cuadra con la tabla"
al_principio = curvas["tfidf_lr"][np.argmin(abs(lt.RECALL - 0.1))]
assert al_principio > 2 * curvas["trivial"][0], "al principio TF-IDF + LR ya no duplica la prevalencia"
assert (curvas["refund"] > curvas["trivial"] + 1e-9).mean() < 0.2, "la regla ya alcanza más de una fracción chica de las tempranas"
assert all(abs(c[-1] - curvas["trivial"][-1]) < 0.05 for c in curvas.values()), "al final las curvas ya no llegan a la prevalencia"
print(f"precisión de TF-IDF + LR al encontrar el 10 % de las tempranas: {al_principio:.2f} (prevalencia {curvas['trivial'][0]:.2f})")

precisión de TF-IDF + LR al encontrar el 10 % de las tempranas: 0.57 (prevalencia 0.16)


**Qué vemos.** TF-IDF + LR queda arriba en casi todo el rango, y donde más se nota es al principio: entre las
reseñas que señala con más seguridad, la parte de tempranas es más del doble de la prevalencia. La regla «refund»
solo alcanza una fracción chica de las tempranas; después su precisión cae a la del trivial, porque la mayoría no
menciona el reembolso. Para encontrar casi todas las tempranas hay que señalar casi todo, y cualquier modelo
termina en la prevalencia.

### 6.3 Si se revisara solo una parte

**Pregunta:** si alguien leyera solo el 10, 20 o 30 % de las negativas con el score más alto, ¿qué parte serían
tempranas? Se calcula en cada fold y se promedia; los empates se rompen por id. La regla «refund» no entra: su
score es 0 o 1 y marca a muy pocas negativas, así que no ordena dentro de esos grupos.

In [20]:
top_k = lt.tabla_top_k(y, {nombre: scores[nombre] for nombre in tx.CANDIDATOS}, fold, negativas["recommendationid"].to_numpy())
display(top_k.round(3))
print(f"la regla «refund» marca al {scores['refund'].mean():.1%} de las negativas")
# Lo que dice el texto de arriba y el de abajo.
assert scores["refund"].mean() < 0.10, "la regla ya marca al 10 % o más de las negativas"
lr_top = top_k["TF-IDF + LR"]
assert 0.35 < lr_top.iloc[0] < 0.5 and lr_top.iloc[0] > 2 * top_k["prevalencia"].iloc[0], "el 10 % ya no es «unas dos de cada cinco»"
assert lr_top.is_monotonic_decreasing, "al ampliar la revisión ya no baja hacia la prevalencia"
assert (lr_top >= top_k[["TF-IDF + NB", "MiniLM + LR"]].max(axis=1)).all(), "TF-IDF + LR ya no queda arriba en los tres cortes"

,prevalencia,TF-IDF + NB,TF-IDF + LR,MiniLM + LR,veces la prevalencia (TF-IDF + LR)
10% con score más alto,0.161,0.240,0.430,0.384,2.676
20% con score más alto,0.161,0.237,0.359,0.340,2.236
30% con score más alto,0.161,0.227,0.308,0.299,1.917


la regla «refund» marca al 4.0% de las negativas


**Qué vemos.** En el 10 % de las negativas que TF-IDF + LR señala primero, las tempranas son unas dos de cada
cinco: más del doble de la prevalencia. Mientras más se amplía la revisión, más se acerca a la prevalencia.
TF-IDF + LR queda arriba de los otros dos modelos en los tres cortes.

### 6.4 Qué palabras pesan

**Pregunta:** ¿qué palabras empujan una reseña hacia temprana o hacia tardía?

TF-IDF + LR se vuelve a ajustar con el entrenamiento de cada fold (el mismo modelo que decide) y se guardan sus
coeficientes. Entran solo los términos (palabras o pares de palabras) que están en los 5 folds y empujan hacia el
mismo lado en los 5. Son asociaciones del modelo, no causas: una palabra pesa porque aparece más en una clase, no
porque explique el arrepentimiento.

Las nubes tienen forma de control: entran todas las palabras que caben, y la de más peso (su coeficiente medio) es
la más grande. Para el dibujo se quitan tres cosas, sin tocar ningún coeficiente:
- `duracion`, la marca de la máscara, y los pares que la llevan: se reportan aparte en la celda de abajo;
- los términos hechos solo de palabras vacías («and the», «your»), con las mismas stopwords del 00, que conservan
  las negaciones («not fun» sí entra);
- una lista declarada de términos ofensivos o fuera de tema (`lt.FUERA_DEL_DIBUJO`), que se imprime con lo que
  quitó.

In [21]:
coeficientes = lt.coeficientes_por_fold(negativas["texto_modelo"], y, appid)
estables = lt.palabras_estables(coeficientes)
duracion_sola, con_duracion = lt.duracion_aparte(coeficientes, estables)
print(f"términos en algún fold: {len(coeficientes):,} · en los 5 y hacia el mismo lado: {len(estables):,}")
display(duracion_sola.round(3).to_frame("«duracion» sola").T)
display(con_duracion.to_frame().T)
# Lo que dice el texto de la sección: «duracion» sola cambia de lado según el fold; sus pares estables van hacia los dos.
assert len(set(duracion_sola.dropna().apply(lambda c: c > 0))) == 2, "«duracion» sola ya empuja hacia el mismo lado en los 5 folds"
assert (con_duracion > 0).all(), "los pares con «duracion» ya no van hacia los dos lados"

términos en algún fold: 36,857 · en los 5 y hacia el mismo lado: 12,775


,fold 1,fold 2,fold 3,fold 4,fold 5
«duracion» sola,-0.034,0.353,0.396,0.539,-0.03


,hacia temprana,hacia tardía
"términos con duracion, estables en los 5 folds",31,46


In [22]:
hacia_temprana, hacia_tardia, fuera_del_dibujo = lt.para_las_nubes(estables)
print(f"fuera del dibujo por la lista declarada ({len(fuera_del_dibujo)}): " + ", ".join(fuera_del_dibujo.index))
nube_temprana, cupieron_temprana = lt.figura_nube(hacia_temprana, lt.COLOR_NUBE["temprana"], "Palabras que empujan hacia temprana")
nube_temprana.show()
nube_tardia, cupieron_tardia = lt.figura_nube(hacia_tardia, lt.COLOR_NUBE["tardía"], "Palabras que empujan hacia tardía")
nube_tardia.show()
print(f"caben {cupieron_temprana:,} de {len(hacia_temprana):,} términos hacia temprana "
      f"y {cupieron_tardia:,} de {len(hacia_tardia):,} hacia tardía")
# Lo que dice el texto de arriba y el de abajo.
primeras_tempranas, primeras_tardias = list(hacia_temprana)[:25], list(hacia_tardia)[:30]
assert {"refunded", "tutorial", "account", "login"} <= set(primeras_tempranas[:20]), "cambió lo que más empuja hacia temprana"
assert {"performance", "stuttering", "crashed", "graphics"} <= set(primeras_tempranas), "el rendimiento ya no está arriba"
assert {"mods", "update", "boss", "dlc"} <= set(primeras_tardias), "cambió lo que más empuja hacia tardía"
assert not any(lt.es_palabra_vacia(t) or lt.va_fuera_del_dibujo(t) for t in [*hacia_temprana, *hacia_tardia]), "se coló una palabra que no va"
assert min(cupieron_temprana, cupieron_tardia) > 300, "ya caben pocas palabras en el control"

fuera del dibujo por la lista declarada (24): cancer, sex, hate you, this ass, piss, racist, wtf, the ass, political, suicide, pissed off, aids, ass and, your ass, politics, ass with, pisses, stupid ass, pisses me, bastard, dei, wtf is, my ass, woke


caben 673 de 2,738 términos hacia temprana y 574 de 7,745 hacia tardía


**Qué vemos.** Hacia temprana pesan tres cosas:
- el reembolso: `refunded`, `refund`;
- lo que pasa al abrir el juego: `tutorial`, `account`, `login`, `settings`;
- el rendimiento: `performance`, `stuttering`, `crashed`, `graphics`.

Hacia tardía pesan las cosas de quien ya jugó mucho: `mods`, `update`, `boss`, `dlc`. En el control caben cientos de
palabras por lado; las chicas son la cola, con menos peso, y dan textura más que lectura.

`duracion` sola no empuja siempre hacia el mismo lado, así que tampoco habría entrado al dibujo. Los pares que la
llevan sí son estables, y hacia los dos lados (las tablas de arriba): la marca dice que hubo una duración, y su
contexto («first duracion», «have duracion») todavía orienta un poco.

### 6.5 Aciertos y errores con score alto

**Pregunta:** ¿cómo se ven las reseñas que el modelo señala con más seguridad? Son las tres tempranas (aciertos) y
las tres tardías (errores) con el score más alto de TF-IDF + LR, fuera de fold, entre las de 8 palabras o más. El
texto es el original; el modelo leyó su versión normalizada y con las duraciones enmascaradas. Son ejemplos, no una
muestra representativa.

In [23]:
juegos_v1, _ = cargar_release(ruta_v1)
ejemplos = lt.ejemplos_con_score_alto(negativas, scores["tfidf_lr"], fold, limpio, juegos_v1.set_index("appid")["nombre"])
display(ejemplos)
# Lo que dice el texto de abajo.
aciertos, errores = ejemplos[ejemplos["caso"].str.startswith("acierto")], ejemplos[ejemplos["caso"].str.startswith("error")]
assert (ejemplos["percentil del score"] > 0.99).all(), "los ejemplos ya no están en el 1 % de score más alto"
assert aciertos["texto"].str.contains("launcher|account|tutorial|crash|refund", case=False).all()
assert errores["texto"].str.contains("refund|tutorial|worth|buy", case=False).all()
assert (errores["minutos jugados"] < 240).all(), "algún error ya pasa de cuatro horas de juego"

,caso,juego,minutos jugados,percentil del score,texto
0,acierto: temprana,DOOM Eternal,53,0.999407,"No Bethesda, I don't care about your launcher, online gamemod, unnecessary tutorial and making an account. Let me play the actual game, thanks. Edit: refunded because the game keep crashing due to..."
1,acierto: temprana,Risk of Rain 2,95,0.999111,"Just not for me. Bought it and refunded it, tried again to see if the hype was true. It is not."
2,acierto: temprana,Stellaris,7,1.000000,couldn't even get past the launcher. game crashes everytime
3,error: tardía,Suicide Squad: Kill the Justice League,218,0.999675,"Terrible gameplay. Not worth to buy, not even on sale."
4,error: tardía,The Lord of the Rings: Gollum™,167,0.998222,Got this for free and I want a refund
5,error: tardía,Dwarf Fortress,216,0.997333,not the games fault but i couldn't get passed the tutorial.


**Qué vemos.** Los aciertos son lo que se espera de una queja temprana: el launcher, la cuenta, el tutorial, que el
juego truene, el reembolso. Los errores se leen igual: piden reembolso, no pasan del tutorial o dicen que no vale
la pena comprarlo. Son tardías por poco, porque ninguna pasa de cuatro horas de juego. Por su texto parecen
tempranas, y lo que las separa es un umbral de minutos que el texto no siempre dice.

### 6.6 Qué le dice esto a NexPlay

**Pregunta:** ¿qué se lleva el negocio de todo esto, y qué cambia en el sitio? La tabla cruza las 40 palabras que más
empujan hacia temprana (las más grandes de la nube roja) con las seis categorías de motivos que muestra hoy el sitio
(`backend/analisis/motivos.py`, las mismas palabras clave de `/explicacion`). «—» quiere decir que el sitio no la
cuenta en ningún motivo.

In [24]:
cobertura = lt.cobertura_del_sitio(hacia_temprana)
display(cobertura.round(3))
sin_categoria = cobertura.index[cobertura["categoría del sitio"] == "—"]
print(f"de las {len(cobertura)} palabras que más empujan hacia temprana, {len(sin_categoria)} no caen en ninguna categoría del sitio")
# Lo que dice la conclusión de abajo.
assert len(sin_categoria) >= 30, "ya no es la gran mayoría la que el sitio no cuenta"
assert {"account", "login", "tutorial", "settings", "performance", "graphics"} <= set(sin_categoria), "el sitio ya cuenta alguna"
assert {"stuttering", "crashed", "fps"} <= set(cobertura.index[cobertura["categoría del sitio"] == "rendimiento"])
assert decision["rama"] == 3, "«Cómo lo usa el sitio» se apoya en la rama 3 de la sección 5"

,coeficiente medio,categoría del sitio
refunded,4.999,—
tutorial,3.853,—
account,3.425,—
graphics,2.848,—
login,2.365,—
performance,2.202,—
looks,2.154,—
stuttering,2.025,rendimiento
the tutorial,1.910,—
settings,1.902,—


de las 40 palabras que más empujan hacia temprana, 35 no caen en ninguna categoría del sitio


**Qué aprendimos.** Quien deja una reseña negativa antes de las dos horas no se queja de lo mismo que quien la deja
después. Las tempranas hablan de entrar al juego (cuenta, login, tutorial, menús, ajustes), del rendimiento en su
equipo (performance, gráficos, tirones, fps, cierres) y del reembolso. Las tardías hablan de actualizaciones, mods,
jefes y DLC.

**Cómo lo usa el sitio.** La ficha de cada juego y la sección «Antes de pagar, esto importa» del Inicio muestran los
motivos de las reseñas con señal de arrepentimiento temprano. La sección 5 decidió que esas quejas tienen lenguaje
propio (rama 3), así que el sitio puede presentarlos como propios de las primeras horas, y no como quejas generales.

**Lo que el sitio todavía no cuenta.** De las 40 palabras que más empujan hacia temprana, la gran mayoría no cae en
ninguna de las seis categorías de la ficha (la cifra, en la celda de arriba). Entre ellas están `account`, `login`,
`tutorial`, `settings`, `performance` y `graphics`, así que hoy esas quejas quedan sin motivo en la ficha. Es el
siguiente paso concreto para el negocio: ampliar las categorías, por ejemplo con «cuenta y acceso» y «primeros
pasos», y sumar `performance` y `graphics` a rendimiento. Necesita su propio prerregistro, como la Parte B, porque
cambia lo que ve la gente.

**Lo que no hace.** El texto no entra al score de riesgo ni predice la experiencia de nadie: el riesgo sigue saliendo
solo de datos del juego.